Statistical analysis of raw data distribution, analysis of missing data, and detection of distribution drift between "Train/Test" using Adversarial Validation.

# **STAGE 1**

In [1]:
import sys
import os
import importlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath("../scripts"))

try:
    import config
    importlib.reload(config)
    from config import TRAIN_CSV, TEST_CSV, PATIENTS_CSV, CLINICS_CSV, VAL_SPLIT_DATE, STAGE1_OUT

    import data_loader
    importlib.reload(data_loader)

    from utils import validation
    importlib.reload(validation)

    print("Success: All modules reloaded from source.")
except Exception as e:
    print(f"Error during manual reload: {e}")

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

Success: All modules reloaded from source.


In [2]:
# Load all raw CSV files
df_train_raw = data_loader.load_raw_data(TRAIN_CSV)
df_test_raw = data_loader.load_raw_data(TEST_CSV)
df_patients = data_loader.load_raw_data(PATIENTS_CSV)
df_clinics = data_loader.load_raw_data(CLINICS_CSV)

# Perform static joins
df_train = data_loader.merge_with_lookups(df_train_raw, df_patients, df_clinics)
df_test = data_loader.merge_with_lookups(df_test_raw, df_patients, df_clinics)

print(f"Initial Train Shape: {df_train.shape}")
print(f"Initial Test Shape: {df_test.shape}")

Initial Train Shape: (200000, 39)
Initial Test Shape: (80000, 38)


In [3]:
# Cleaning and Downcasting
for df in [df_train, df_test]:
    df = data_loader.convert_to_datetime(df, ['appointment_datetime', 'booking_datetime'])
    # Fix time leakage (Lead_Time check)
    df = data_loader.handle_lead_time_leakage(df)
    # Apply SMS logic and flags
    df = data_loader.process_sms_logic(df)
    df = data_loader.downcast_memory(df)

print(f"Min Lead Time in Train: {df_train['lead_time_hours'].min()}")

Min Lead Time in Train: 0.5000413060188293


# **STAGE 2**

In [4]:
# Sub-Train: Jan-Sep 2025 | Sub-Val: Oct 2025
sub_train, sub_val = validation.split_by_time(df_train, VAL_SPLIT_DATE)

print(f"Sub-Train Range: {sub_train['appointment_datetime'].min()} to {sub_train['appointment_datetime'].max()}")
print(f"Sub-Val Range: {sub_val['appointment_datetime'].min()} to {sub_val['appointment_datetime'].max()}")

Sub-Train Range: 2025-01-01 08:00:00 to 2025-09-30 17:45:00
Sub-Val Range: 2025-10-01 08:00:00 to 2025-10-31 17:45:00


In [6]:
check_features = ['age', 'ses_score', 'lead_time_hours', 'sms_lead_hours']

adv_data = validation.prepare_adversarial_data(sub_train, df_test, check_features)

auc_score, feat_imp = validation.run_adversarial_validation(adv_data, check_features)

print(f"Adversarial AUC: {auc_score:.4f}")

# Visualize feature differences if AUC > 0.70
if auc_score > 0.70:
    print("Significant distribution shift detected")
    plt.figure(figsize=(10, 5))
    feat_imp.plot(kind='barh', color='salmon')
    plt.title("Features causing Train/Test deviation")
    plt.show()

Adversarial AUC: 0.5016


In [8]:
# Save to processed_data/stage1
sub_train.to_parquet(STAGE1_OUT / "train_base.parquet", index=False)
sub_val.to_parquet(STAGE1_OUT / "val_base.parquet", index=False)
df_test.to_parquet(STAGE1_OUT / "test_base.parquet", index=False)